In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import pickle
import torch

import sys
sys.path.insert(0, str(Path.cwd().parents[0] / '2_Propensities'))
import SASRec_class as sasrec

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

## 1 Load Processed Data

In [ ]:
base_artifacts = Path.cwd().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Simulation'
data = pd.read_csv(
    data_path / 'simulation_data.csv'
)
print(f"Data shape: {data.shape}")

with open(data_path / 'test_users.pkl', 'rb') as f:
    test_users = pickle.load(f)

unique_users = data['user_id'].unique()
train_users = [user for user in unique_users if user not in test_users]

Data shape: (750000, 3)


In [3]:
# History length
L = 50

# 2 Prepare Windows for Training

In [4]:
users_dict = data.groupby('user_id')['item_id'].apply(list).to_dict()

In [5]:
padding_idx = data['item_id'].max() + 1

train_dataset = []
test_dataset = []
for user_id in tqdm(unique_users):
    padded_sequence = [padding_idx] * (L - 2) + users_dict[user_id]
    for i in range(len(padded_sequence) - L + 1):
        window = padded_sequence[i:i+L]
        if user_id in train_users:
            train_dataset.append(window)
        else:
            test_dataset.append(window)

train_dataset = np.array(train_dataset)
test_dataset = np.array(test_dataset)

  0%|          | 0/5000 [00:00<?, ?it/s]

# 3 Train the model

In [6]:
model = sasrec.SASRecTorch(
    num_items=padding_idx+1,
    max_seq_len=L,
    d_model=50,
    n_heads=1,
    n_layers=2,
    dropout=0.5,
    device="cuda",
)
model.fit(
    train_dataset=train_dataset,
    valid_dataset=test_dataset,
    batch_size=2**11,
    lr=1e-3,
    weight_decay=0.0, 
    num_epochs=10,
)

/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch | T-Loss | V-Loss | Pctl  | HR10  | NDCG  | Cosθ  | Elapsed Time
======|========|========|=======|=======|=======|=======|=============
    1 |  1.219 |  0.787 | 0.884 | 0.652 | 0.406 | None  |     00:19.0
    2 |  0.695 |  0.569 | 0.924 | 0.898 | 0.530 | 0.492 |     00:37.5
    3 |  0.565 |  0.522 | 0.926 | 0.902 | 0.535 | 0.500 |     00:55.8
    4 |  0.520 |  0.509 | 0.926 | 0.902 | 0.534 | 0.675 |     01:14.0
    5 |  0.499 |  0.503 | 0.927 | 0.903 | 0.535 | 0.732 |     01:32.3
    6 |  0.486 |  0.500 | 0.927 | 0.903 | 0.534 | 0.716 |     01:47.6
    7 |  0.477 |  0.499 | 0.927 | 0.903 | 0.535 | 0.687 |     02:00.7
    8 |  0.470 |  0.499 | 0.927 | 0.903 | 0.534 | 0.642 |     02:13.9
    9 |  0.466 |  0.498 | 0.927 | 0.904 | 0.534 | 0.583 |     02:27.0
   10 |  0.462 |  0.498 | 0.927 | 0.904 | 0.534 | 0.521 |     02:40.2


In [7]:
folder_path = base_artifacts / 'SASRec_Models' / 'simulation'
model.save(path=folder_path / f'sasrec.pt')

init_dict = {
    "num_items": padding_idx+1,
    "max_seq_len": L,
    "d_model": model.d_model,
    "n_heads": model.n_heads,
    "n_layers": model.n_layers,
    "dropout": model.dropout,
    "device": model.device
}

with open(folder_path / f'init_dict.pkl', 'wb') as f:
    pickle.dump(init_dict, f)

# 4 Load a Model

In [8]:
folder_path = base_artifacts / 'SASRec_Models' / 'simulation'
with open(folder_path / f'init_dict.pkl', 'rb') as f:
    init_dict_loaded = pickle.load(f)

loaded_model = sasrec.SASRecTorch(**init_dict_loaded)
loaded_model.load(folder_path / f'sasrec.pt')

Model loaded from /home/gouni/CausalI2I_new_artifacts/SASRec_Models/simulation/sasrec.pt.
num_items:     3001
max_seq_len:   50
device:        cuda
batch_size:    2048
lr:            0.001
weight_decay:  0.0
num_epochs:    10
saved_at:      2026-08-15 09:34:05
note:          None


/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
